### RMSNorm 公式

RMSNorm（Root Mean Square Normalization，均方根归一化）的标准形式为：

$$
\operatorname{RMSNorm}(x)
=
\gamma \cdot
\frac{x}
{\sqrt{
\frac{1}{d}
\sum_{i=1}^{d}x_i^2
+
\epsilon
}}
$$

其中：

> `d`：当前 token 的特征维度数。例如 MiniMind 中特征维度为 768 时，`d = 768`。

也可以定义：

$$
\operatorname{RMS}(x)
=
\sqrt{
\epsilon+
\frac{1}{d}
\sum_{i=1}^{d}x_i^2
}
$$

于是：

$$
\operatorname{RMSNorm}(x)
=
\gamma\cdot
\frac{x}{\operatorname{RMS}(x)}
$$

代码中的等价形式为：

$$
\operatorname{RMSNorm}(x)
=
x\cdot
\operatorname{rsqrt}
\left(
\operatorname{mean}(x^2)+\epsilon
\right)
\cdot\gamma
$$

参数说明：

* γ：可学习的缩放参数，每个特征维度都有一个对应的缩放值；
* ε：一个极小的正数，用于保证数值稳定；
* `mean(x²)`：对当前 token 的所有特征维度平方后求平均；
* `rsqrt(z)`：表示 `1 / √z`。

对应 PyTorch 代码：

```python
x * torch.rsqrt(
    x.pow(2).mean(dim=-1, keepdim=True) + eps
) * weight
```

其中，`weight` 就对应公式中的可学习缩放参数 γ。


In [11]:
# 手写一个RMSNorm
import torch

def my_rmsnorm(x, weight, eps=1e-6):
    # 1. 平方
    x_squared = x.pow(2)

    # 2. 最后一维求均值
    mean_square = x_squared.mean(dim=-1, keepdim=True)

    # 3. 开根号得到 RMS
    rms = torch.sqrt(mean_square + eps)

    # 4. 归一化
    x_norm = x / rms

    # 5. 乘可学习缩放参数
    return x_norm * weight # weight就是缩放参数"γ"
#特征向量 x 负责“携带当前这个 token 的信息”，γ 负责“学习每个特征维度应该被放大还是缩小多少”

In [ ]:
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [10.0, 20.0, 30.0]
])

weight = torch.ones(3)
print(weight.shape)

y = my_rmsnorm(x, weight)

print(y)
print(y.pow(2).mean(dim=-1)) 
# dim=-1：沿最后一个维度求平均，也就是对当前 token 的所有特征维度求平均

torch.Size([3])
tensor([[0.4629, 0.9258, 1.3887],
        [0.4629, 0.9258, 1.3887]])
tensor([1.0000, 1.0000])


# RoPE


In [ ]:
222